In [1]:
import pandas as pd
from tqdm import tqdm

In [2]:
val_path = "data/global_split/reviews_Beauty_5/validation.csv"
train_path = "data/global_split/reviews_Beauty_5/train.csv"
test_path = "data/global_split/reviews_Beauty_5/test.csv"
val = pd.read_csv(val_path)
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)


In [18]:
(train.groupby(by="user_id").count()["item_id"] < 3).sum()

np.int64(1646)

In [ ]:
df = (train.groupby(by="user_id").count()["item_id"] < 3)
bad_users = df[df].index

Index([   11,    16,    18,    39,    61,    73,    78,   101,   109,   116,
       ...
       22153, 22178, 22205, 22246, 22265, 22275, 22276, 22289, 22291, 22318],
      dtype='int64', name='user_id', length=1646)

In [ ]:
validation = train.groupby(by="user_id").last().reset_index()

,user_id,item_id,rating,timestamp
0,1,1,1,1391040000
1,3,243,1,1393545600
2,4,1,1,1386460800
3,5,11444,1,1384992000
4,6,5664,1,1392422400
...,...,...,...,...
20218,22333,11810,1,1390435200
20219,22334,11941,1,1392768000
20220,22335,11941,1,1393632000
20221,22336,11941,1,1394064000


In [7]:
train

,user_id,item_id,rating,timestamp
0,225,104,1,1023840000
1,225,101,1,1024185600
2,225,102,1,1024185600
3,1115,152,1,1036627200
4,225,23,1,1052611200
...,...,...,...,...
138162,13627,11768,1,1394150400
138163,15343,11768,1,1394150400
138164,3541,11770,1,1394150400
138165,2670,11774,1,1394150400


In [8]:
len(set(test.user_id).difference(set(train.user_id)))

2140

In [14]:
df

,user_id,item_id,rating,timestamp
0,1,5406,1,1391040000
1,3,11204,1,1393804800
2,4,7849,1,1386460800
3,5,11390,1,1388966400
4,6,7595,1,1392422400
...,...,...,...,...
20218,22333,11893,1,1392681600
20219,22334,11942,1,1392768000
20220,22335,11966,1,1393632000
20221,22336,11966,1,1394064000


In [18]:
data_path = "data/global_split/reviews_Beauty_5/validation.csv"
all_data_path = "data/reviews_Beauty_5.csv"
df = pd.read_csv(data_path)
all_data = pd.read_csv(all_data_path)
max_sequence_length = 10
index = []
for _, (val_user_id, val_item_id, val_rating, val_timestamp) in tqdm(
    df.iterrows(), total=len(df), desc="Validation ds creation"
):
    user_info = all_data[all_data.user_id == val_user_id]
    previous_user_info = user_info[user_info.timestamp <= val_timestamp].copy()
    # to make out item last
    previous_user_info.loc[previous_user_info.user_id == val_user_id, "timestamp"] += 1
    previous_user_info = previous_user_info.reset_index().sort_values(
        by=["timestamp", "index"]
    )
    previous_user_items = previous_user_info.item_id.tolist()
    assert previous_user_items[-1] == val_item_id
    item_sequence = previous_user_items[-max_sequence_length :]
    index.append(
        {
            "user.ids": [val_user_id],
            "user.length": 1,
            "item.ids": item_sequence,
            "item.length": len(item_sequence),
        }
    )

Validation ds creation: 100%|██████████| 20223/20223 [00:12<00:00, 1622.51it/s]


In [19]:
index

[{'user.ids': [1],
  'user.length': 1,
  'item.ids': [6846, 7873, 4585, 1, 5406],
  'item.length': 5},
 {'user.ids': [3],
  'user.length': 1,
  'item.ids': [1, 6050, 7977, 5252, 4211, 243, 11204],
  'item.length': 7},
 {'user.ids': [4],
  'user.length': 1,
  'item.ids': [5522, 439, 5161, 11140, 1, 7849],
  'item.length': 6},
 {'user.ids': [5],
  'user.length': 1,
  'item.ids': [1, 10470, 10064, 9403, 10362, 4758, 6500, 11444, 11390],
  'item.length': 9},
 {'user.ids': [6],
  'user.length': 1,
  'item.ids': [8816, 4595, 9917, 6046, 9597, 8819, 6431, 9562, 5664, 7595],
  'item.length': 10},
 {'user.ids': [7],
  'user.length': 1,
  'item.ids': [8499, 8778, 1796, 5987, 6059, 10038, 1],
  'item.length': 7},
 {'user.ids': [8],
  'user.length': 1,
  'item.ids': [8997, 4554, 6928, 1, 2367],
  'item.length': 5},
 {'user.ids': [9],
  'user.length': 1,
  'item.ids': [3697, 5488, 11243, 11288, 10357, 3894, 4352, 9557, 3362, 3788],
  'item.length': 10},
 {'user.ids': [10],
  'user.length': 1,
  'it

In [6]:
df = pd.read_csv(data_path)

In [10]:
user_items = df.groupby("user_id")["item_id"].apply(list).to_dict()
index = []
max_sequence_length = 10
for user_idx, item_ids in sorted(list(user_items.items()), key=lambda x: x[0]):
    index.append(
        {
            "user.ids": [user_idx],
            "user.length": 1,
            "item.ids": item_ids[-max_sequence_length :],
            "item.length": len(item_ids[-max_sequence_length :]),
        }
    )

In [ ]:
for user_idx, item_ids in sorted(list(user_items.items()), key=lambda x: x[0]):

{1: [6846, 7873, 4585, 1],
 3: [1, 6050, 7977, 5252, 4211, 243],
 4: [5522, 439, 5161, 11140, 1],
 5: [1, 10470, 10064, 9403, 10362, 4758, 6500, 11444],
 6: [6906,
  7241,
  5662,
  1,
  411,
  5133,
  6404,
  8816,
  4595,
  9917,
  6046,
  9597,
  8819,
  6431,
  9562,
  5664],
 7: [8499, 8778, 1796, 5987, 6059, 10038],
 8: [8997, 4554, 6928, 1],
 9: [373,
  4488,
  6865,
  5161,
  5920,
  7454,
  3974,
  1203,
  6236,
  11095,
  9383,
  2722,
  3697,
  5488,
  11243,
  11288,
  10357,
  3894,
  4352,
  9557,
  3362],
 10: [1928, 2171, 2228, 2],
 11: [1190],
 12: [4211, 3275, 6325, 9046, 7591, 2, 8643],
 13: [6322, 5528, 7437, 10874],
 14: [2, 138, 2937, 5128, 8517, 1307, 8327],
 15: [11518, 8579, 11256, 2, 1305, 2047],
 16: [6276],
 17: [6534, 8958, 2, 2421],
 18: [11111],
 19: [5244, 8815, 2],
 20: [6067,
  5404,
  7448,
  7462,
  7798,
  771,
  1094,
  3228,
  4258,
  1697,
  2746,
  2949,
  5318,
  7172,
  1053,
  1962,
  2400,
  2506,
  2835,
  3890,
  8104,
  3,
  6778,
  2998,